In [ ]:
import pandas as pd
import requests
import time
from datetime import datetime

# Caminho do arquivo .xls
arquivo = r"D:\Cotic\CONFERE_CUSTODIADOS_20_02_2026 10_14_48.xls"

# Lê a planilha usando xlrd 
df = pd.read_excel(arquivo, sheet_name="confereCustodiado", engine="xlrd")

# Lê a planilha
#df = pd.read_excel(arquivo, sheet_name="confereCustodiado")

# Extrai os IDs da primeira coluna (Prontuário)
#ids = df.iloc[:, 0].dropna().astype(int).tolist()
# Extrai e formata os IDs da primeira coluna (Prontuário)

ids_raw = df.iloc[:, 0].astype(str).str.strip()  # força string e remove espaços

# Remove valores vazios ou não numéricos
ids_clean = []
for val in ids_raw:
    if val and val.isdigit():  # só aceita números puros
        ids_clean.append(int(val))

ids = ids_clean
print(f"Total de IDs capturados: {len(ids)}")
print(ids[:10])  # mostra os 10 primeiros para conferência

# Token de autorização (copie do DevTools)
TOKEN = "Bearer eyJhbGciOiJIUzI1NiJ9.eyJzdWIiOiJlbGlzYW5nZWxhLmhlbGNpYXMiLCJhdXRoIjpbeyJhdXRob3JpdHkiOiJBRFZPR0FET18tX0FCQV9SRUxBQ09FU19BTFRFUkFSIn0seyJhdXRob3JpdHkiOiJBRFZPR0FET19BTFRFUkFSIn0seyJhdXRob3JpdHkiOiJBRFZPR0FET19DT05TVUxUQVIifSx7ImF1dGhvcml0eSI6IkFEVk9HQURPX0lNUFJJTUlSIn0seyJhdXRob3JpdHkiOiJBR0VOREFNRU5UT19KVVJJRElDT19BTFRFUkFSIn0seyJhdXRob3JpdHkiOiJBR0VOREFNRU5UT19KVVJJRElDT19DT05TVUxUQVIifSx7ImF1dGhvcml0eSI6IkFHRU5EQU1FTlRPX0pVUklESUNPX0lNUFJJTUlSIn0seyJhdXRob3JpdHkiOiJBR0VOREFNRU5UT19KVVJJRElDT19JTlNFUklSIn0seyJhdXRob3JpdHkiOiJBR0VOREFNRU5UT19TT0NJQUxfQ09OU1VMVEFSIn0seyJhdXRob3JpdHkiOiJBR0VOREFNRU5UT19TT0NJQUxfSU1QUklNSVIifSx7ImF1dGhvcml0eSI6IkFVVE9SSVpBUl9BQ0VTU09fQUxURVJBUiJ9LHsiYXV0aG9yaXR5IjoiQVVUT1JJWkFSX0FDRVNTT19BVVRPUklaQVIifSx7ImF1dGhvcml0eSI6IkFVVE9SSVpBUl9BQ0VTU09fQ09OU1VMVEFSIn0seyJhdXRob3JpdHkiOiJBVVRPUklaQVJfQUNFU1NPX0lNUFJJTUlSIn0seyJhdXRob3JpdHkiOiJBVVRPUklaQVJfQUNFU1NPX0lOU0VSSVIifSx7ImF1dGhvcml0eSI6IkFVVE9SSVpBUl9CSU9NRVRSSUFfQUxURVJBUiJ9LHsiYXV0aG9yaXR5IjoiQVVUT1JJWkFSX0JJT01FVFJJQV9BVVRPUklaQVIifSx7ImF1dGhvcml0eSI6IkFVVE9SSVpBUl9CSU9NRVRSSUFfQ09OU1VMVEFSIn0seyJhdXRob3JpdHkiOiJBVVRPUklaQVJfQklPTUVUUklBX0lNUFJJTUlSIn0seyJhdXRob3JpdHkiOiJBVVRPUklaQVJfQklPTUVUUklBX0lOU0VSSVIifSx7ImF1dGhvcml0eSI6IkJMT1FVRUlPX0FDRVNTT19DT05TVUxUQVIifSx7ImF1dGhvcml0eSI6IkJMT1FVRUlPX0FDRVNTT19JTlNFUklSIn0seyJhdXRob3JpdHkiOiJCTE9RVUVJT19BTFRFUkFSIn0seyJhdXRob3JpdHkiOiJCTE9RVUVJT19DT05TVUxUQVIifSx7ImF1dGhvcml0eSI6IkJMT1FVRUlPX0lOU0VSSVIifSx7ImF1dGhvcml0eSI6IkNBREFTVFJPX1BST0NFU1NPU19BTFRFUkFSIn0seyJhdXRob3JpdHkiOiJDQURBU1RST19QUk9DRVNTT1NfQ09OU1VMVEFSIn0seyJhdXRob3JpdHkiOiJDQURBU1RST19QUk9DRVNTT1NfSU1QUklNSVIifSx7ImF1dGhvcml0eSI6IkNBREFTVFJPX1BST0NFU1NPU19JTlNFUklSIn0seyJhdXRob3JpdHkiOiJDT05GRVJFX0NPTlNVTFRBUiJ9LHsiYXV0aG9yaXR5IjoiQ09ORkVSRV9JTVBSSU1JUiJ9LHsiYXV0aG9yaXR5IjoiQ1VTVE9ESUFET18tX1RFTEVGT05FX0VYQ0xVSVIifSx7ImF1dGhvcml0eSI6IkNVU1RPRElBRE9fQUxURVJBUiJ9LHsiYXV0aG9yaXR5IjoiQ1VTVE9ESUFET19DT05TVUxUQVIifSx7ImF1dGhvcml0eSI6IkNVU1RPRElBRE9fSU1QUklNSVIifSx7ImF1dGhvcml0eSI6IkNVU1RPRElBRE9fSU5TRVJJUiJ9LHsiYXV0aG9yaXR5IjoiREFTSEJPQVJEX0NPTlNVTFRBUiJ9LHsiYXV0aG9yaXR5IjoiRU5UUkFEQV9TRU1fQUdFTkRBTUVOVE9fQUxURVJBUiJ9LHsiYXV0aG9yaXR5IjoiRU5UUkFEQV9TRU1fQUdFTkRBTUVOVE9fQVVUT1JJWkFSIn0seyJhdXRob3JpdHkiOiJFTlRSQURBX1NFTV9BR0VOREFNRU5UT19DT05TVUxUQVIifSx7ImF1dGhvcml0eSI6IkVOVFJBREFfU0VNX0FHRU5EQU1FTlRPX0VYQ0xVSVIifSx7ImF1dGhvcml0eSI6IkVOVFJBREFfU0VNX0FHRU5EQU1FTlRPX0lNUFJJTUlSIn0seyJhdXRob3JpdHkiOiJFTlRSQURBX1NFTV9BR0VOREFNRU5UT19JTlNFUklSIn0seyJhdXRob3JpdHkiOiJFUVVJUEVfQUxURVJBUiJ9LHsiYXV0aG9yaXR5IjoiRVFVSVBFX0NPTlNVTFRBUiJ9LHsiYXV0aG9yaXR5IjoiRVFVSVBFX0lOU0VSSVIifSx7ImF1dGhvcml0eSI6IkVTUEVDSUFMSURBREVfTUVESUNBX0FVVE9SSVpBUiJ9LHsiYXV0aG9yaXR5IjoiRVNQRUNJQUxJREFERV9NRURJQ0FfSU1QUklNSVIifSx7ImF1dGhvcml0eSI6IkVTVEFUSVNUSUNBU19DT05TVUxUQVIifSx7ImF1dGhvcml0eSI6IkZBQ0NBT19BTFRFUkFSIn0seyJhdXRob3JpdHkiOiJGQUNDQU9fSU1QUklNSVIifSx7ImF1dGhvcml0eSI6IkZBQ0NBT19JTlNFUklSIn0seyJhdXRob3JpdHkiOiJGSUNIQV9BRFZPR0FET18tX0JMT1FVRUlPX0FMVEVSQVIifSx7ImF1dGhvcml0eSI6IkZJQ0hBX0FEVk9HQURPXy1fQkxPUVVFSU9fQ09OU1VMVEFSIn0seyJhdXRob3JpdHkiOiJGSUNIQV9BRFZPR0FET18tX0JMT1FVRUlPX0lOU0VSSVIifSx7ImF1dGhvcml0eSI6IkZJQ0hBX0FEVk9HQURPXy1fQ1VTVE9ESUFET19DT05TVUxUQVIifSx7ImF1dGhvcml0eSI6IkZJQ0hBX0FEVk9HQURPXy1fSElTVE9SSUNPX1ZJU0lUQV9DT05TVUxUQVIifSx7ImF1dGhvcml0eSI6IkZJQ0hBX0FEVk9HQURPXy1fVU5JREFERVNfQUNFU1NPX0NPTlNVTFRBUiJ9LHsiYXV0aG9yaXR5IjoiRklDSEFfQURWT0dBRE9fQ09OU1VMVEFSIn0seyJhdXRob3JpdHkiOiJGSUNIQV9BRFZPR0FET19JTVBSSU1JUiJ9LHsiYXV0aG9yaXR5IjoiRklDSEFfQ1VTVE9ESUFET18tX0FEVk9HQURPX0NPTlNVTFRBUiJ9LHsiYXV0aG9yaXR5IjoiRklDSEFfQ1VTVE9ESUFET18tX0FORVhPX0NPTlNVTFRBUiJ9LHsiYXV0aG9yaXR5IjoiRklDSEFfQ1VTVE9ESUFET18tX0FORVhPX0VYQ0xVSVIifSx7ImF1dGhvcml0eSI6IkZJQ0hBX0NVU1RPRElBRE9fLV9BTkVYT19JTVBSSU1JUiJ9LHsiYXV0aG9yaXR5IjoiRklDSEFfQ1VTVE9ESUFET18tX0FORVhPX0lOU0VSSVIifSx7ImF1dGhvcml0eSI6IkZJQ0hBX0NVU1RPRElBRE9fLV9BVElWSURBREVfTEFCT1JBTF9BTFRFUkFSIn0seyJhdXRob3JpdHkiOiJGSUNIQV9DVVNUT0RJQURPXy1fQVRJVklEQURFX0xBQk9SQUxfQ09OU1VMVEFSIn0seyJhdXRob3JpdHkiOiJGSUNIQV9DVVNUT0RJQURPXy1fQVRJVklEQURFX0xBQk9SQUxfSU1QUklNSVIifSx7ImF1dGhvcml0eSI6IkZJQ0hBX0NVU1RPRElBRE9fLV9BVElWSURBREVfTEFCT1JBTF9JTlNFUklSIn0seyJhdXRob3JpdHkiOiJGSUNIQV9DVVNUT0RJQURPXy1fQklPTUVUUklBX0NPTlNVTFRBUiJ9LHsiYXV0aG9yaXR5IjoiRklDSEFfQ1VTVE9ESUFET18tX0JJT01FVFJJQV9JTlNFUklSIn0seyJhdXRob3JpdHkiOiJGSUNIQV9DVVNUT0RJQURPXy1fQ0VSVElEQU9fQ0FSQ0VSQVJJQV9DT05TVUxUQVIifSx7ImF1dGhvcml0eSI6IkZJQ0hBX0NVU1RPRElBRE9fLV9DRVJUSURBT19DQVJDRVJBUklBX0lNUFJJTUlSIn0seyJhdXRob3JpdHkiOiJGSUNIQV9DVVNUT0RJQURPXy1fQ09NUEFOSEVJUk9fQ09OU1VMVEFSIn0seyJhdXRob3JpdHkiOiJGSUNIQV9DVVNUT0RJQURPXy1fQ09NUE9SVEFNRU5UT19BTFRFUkFSIn0seyJhdXRob3JpdHkiOiJGSUNIQV9DVVNUT0RJQURPXy1fQ09NUE9SVEFNRU5UT19DT05TVUxUQVIifSx7ImF1dGhvcml0eSI6IkZJQ0hBX0NVU1RPRElBRE9fLV9DT01QT1JUQU1FTlRPX0lNUFJJTUlSIn0seyJhdXRob3JpdHkiOiJGSUNIQV9DVVNUT0RJQURPXy1fQ09NUE9SVEFNRU5UT19JTlNFUklSIn0seyJhdXRob3JpdHkiOiJGSUNIQV9DVVNUT0RJQURPXy1fQ09ORElDT0VTX0NPTlNVTFRBUiJ9LHsiYXV0aG9yaXR5IjoiRklDSEFfQ1VTVE9ESUFET18tX0ZBTFRBU19BTFRFUkFSIn0seyJhdXRob3JpdHkiOiJGSUNIQV9DVVNUT0RJQURPXy1fRkFMVEFTX0NPTlNVTFRBUiJ9LHsiYXV0aG9yaXR5IjoiRklDSEFfQ1VTVE9ESUFET18tX0ZBTFRBU19JTVBSSU1JUiJ9LHsiYXV0aG9yaXR5IjoiRklDSEFfQ1VTVE9ESUFET18tX0ZBTFRBU19JTlNFUklSIn0seyJhdXRob3JpdHkiOiJGSUNIQV9DVVNUT0RJQURPXy1fRklMSE9fQUxURVJBUiJ9LHsiYXV0aG9yaXR5IjoiRklDSEFfQ1VTVE9ESUFET18tX0ZJTEhPX0NPTlNVTFRBUiJ9LHsiYXV0aG9yaXR5IjoiRklDSEFfQ1VTVE9ESUFET18tX0ZJTEhPX0lOU0VSSVIifSx7ImF1dGhvcml0eSI6IkZJQ0hBX0NVU1RPRElBRE9fLV9HRVNUQU5URV9BTFRFUkFSIn0seyJhdXRob3JpdHkiOiJGSUNIQV9DVVNUT0RJQURPXy1fR0VTVEFOVEVfQ09OU1VMVEFSIn0seyJhdXRob3JpdHkiOiJGSUNIQV9DVVNUT0RJQURPXy1fR0VTVEFOVEVfSU5TRVJJUiJ9LHsiYXV0aG9yaXR5IjoiRklDSEFfQ1VTVE9ESUFET18tX0pVUklESUNPX0NPTlNVTFRBUiJ9LHsiYXV0aG9yaXR5IjoiRklDSEFfQ1VTVE9ESUFET18tX0pVUklESUNPX0lNUFJJTUlSIn0seyJhdXRob3JpdHkiOiJGSUNIQV9DVVNUT0RJQURPXy1fTEFDVEFOVEVfQUxURVJBUiJ9LHsiYXV0aG9yaXR5IjoiRklDSEFfQ1VTVE9ESUFET18tX0xBQ1RBTlRFX0NPTlNVTFRBUiJ9LHsiYXV0aG9yaXR5IjoiRklDSEFfQ1VTVE9ESUFET18tX0xBQ1RBTlRFX0lOU0VSSVIifSx7ImF1dGhvcml0eSI6IkZJQ0hBX0NVU1RPRElBRE9fLV9NRURJREFfRElTQ0lQTElOQVJfQUxURVJBUiJ9LHsiYXV0aG9yaXR5IjoiRklDSEFfQ1VTVE9ESUFET18tX01FRElEQV9ESVNDSVBMSU5BUl9DT05TVUxUQVIifSx7ImF1dGhvcml0eSI6IkZJQ0hBX0NVU1RPRElBRE9fLV9NRURJREFfRElTQ0lQTElOQVJfSU1QUklNSVIifSx7ImF1dGhvcml0eSI6IkZJQ0hBX0NVU1RPRElBRE9fLV9NRURJREFfRElTQ0lQTElOQVJfSU5TRVJJUiJ9LHsiYXV0aG9yaXR5IjoiRklDSEFfQ1VTVE9ESUFET18tX01PVklNRU5UQUNBT19DT05TVUxUQVIifSx7ImF1dGhvcml0eSI6IkZJQ0hBX0NVU1RPRElBRE9fLV9NT1ZJTUVOVEFDQU9fRVhDTFVJUiJ9LHsiYXV0aG9yaXR5IjoiRklDSEFfQ1VTVE9ESUFET18tX1BST0NFU1NPX0FMVEVSQVIifSx7ImF1dGhvcml0eSI6IkZJQ0hBX0NVU1RPRElBRE9fLV9QUk9DRVNTT19DT05TVUxUQVIifSx7ImF1dGhvcml0eSI6IkZJQ0hBX0NVU1RPRElBRE9fLV9QUk9DRVNTT19JTlNFUklSIn0seyJhdXRob3JpdHkiOiJGSUNIQV9DVVNUT0RJQURPXy1fUkVHSVNUUkFSX0RJQVNfUkVNSVNTQU9fQUxURVJBUiJ9LHsiYXV0aG9yaXR5IjoiRklDSEFfQ1VTVE9ESUFET18tX1JFR0lTVFJBUl9ESUFTX1JFTUlTU0FPX0NPTlNVTFRBUiJ9LHsiYXV0aG9yaXR5IjoiRklDSEFfQ1VTVE9ESUFET18tX1JFR0lTVFJBUl9ESUFTX1JFTUlTU0FPX0lNUFJJTUlSIn0seyJhdXRob3JpdHkiOiJGSUNIQV9DVVNUT0RJQURPXy1fUkVHSVNUUkFSX0RJQVNfUkVNSVNTQU9fSU5TRVJJUiJ9LHsiYXV0aG9yaXR5IjoiRklDSEFfQ1VTVE9ESUFET18tX1JFTUlSX0RJQVNfQUxURVJBUiJ9LHsiYXV0aG9yaXR5IjoiRklDSEFfQ1VTVE9ESUFET18tX1JFTUlSX0RJQVNfQ09OU1VMVEFSIn0seyJhdXRob3JpdHkiOiJGSUNIQV9DVVNUT0RJQURPXy1fUkVNSVJfRElBU19JTVBSSU1JUiJ9LHsiYXV0aG9yaXR5IjoiRklDSEFfQ1VTVE9ESUFET18tX1JFTUlSX0RJQVNfSU5TRVJJUiJ9LHsiYXV0aG9yaXR5IjoiRklDSEFfQ1VTVE9ESUFET18tX1NBVURFX0NPTlNVTFRBUiJ9LHsiYXV0aG9yaXR5IjoiRklDSEFfQ1VTVE9ESUFET18tX1NJVFVBQ0FPX0pVUklESUNBX0FMVEVSQVIifSx7ImF1dGhvcml0eSI6IkZJQ0hBX0NVU1RPRElBRE9fLV9TSVRVQUNBT19KVVJJRElDQV9DT05TVUxUQVIifSx7ImF1dGhvcml0eSI6IkZJQ0hBX0NVU1RPRElBRE9fLV9VTklGSUNBQ0FPX05PVkFfQ09OU1VMVEFSIn0seyJhdXRob3JpdHkiOiJGSUNIQV9DVVNUT0RJQURPXy1fVklTSVRBX0NPTlNVTFRBUiJ9LHsiYXV0aG9yaXR5IjoiRklDSEFfQ1VTVE9ESUFET19DT05TVUxUQVIifSx7ImF1dGhvcml0eSI6IkZJQ0hBX0NVU1RPRElBRE9fSU1QUklNSVIifSx7ImF1dGhvcml0eSI6IkZJQ0hBX1BFU1NPQV9EQURPU19QRVNTT0FJU19DT05TVUxUQVIifSx7ImF1dGhvcml0eSI6IkZJQ0hBX1BSRVNUQURPUl9TRVJWSUNPXy1fQkxPUVVFSU9fQUxURVJBUiJ9LHsiYXV0aG9yaXR5IjoiRklDSEFfUFJFU1RBRE9SX1NFUlZJQ09fLV9CTE9RVUVJT19DT05TVUxUQVIifSx7ImF1dGhvcml0eSI6IkZJQ0hBX1BSRVNUQURPUl9TRVJWSUNPXy1fQkxPUVVFSU9fSU5TRVJJUiJ9LHsiYXV0aG9yaXR5IjoiRklDSEFfUFJFU1RBRE9SX1NFUlZJQ09fLV9ISVNUT1JJQ09fVklTSVRBX0NPTlNVTFRBUiJ9LHsiYXV0aG9yaXR5IjoiRklDSEFfUFJFU1RBRE9SX1NFUlZJQ09fLV9VTklEQURFU19BQ0VTU09fQ09OU1VMVEFSIn0seyJhdXRob3JpdHkiOiJGSUNIQV9QUkVTVEFET1JfU0VSVklDT19DT05TVUxUQVIifSx7ImF1dGhvcml0eSI6IkZJQ0hBX1BSRVNUQURPUl9TRVJWSUNPX0lNUFJJTUlSIn0seyJhdXRob3JpdHkiOiJGSUNIQV9WSVNJVEFOVEVfLV9CTE9RVUVJT19BTFRFUkFSIn0seyJhdXRob3JpdHkiOiJGSUNIQV9WSVNJVEFOVEVfLV9CTE9RVUVJT19DT05TVUxUQVIifSx7ImF1dGhvcml0eSI6IkZJQ0hBX1ZJU0lUQU5URV8tX0JMT1FVRUlPX0lOU0VSSVIifSx7ImF1dGhvcml0eSI6IkZJQ0hBX1ZJU0lUQU5URV8tX0NVU1RPRElBRE9fQ09OU1VMVEFSIn0seyJhdXRob3JpdHkiOiJGSUNIQV9WSVNJVEFOVEVfLV9ISVNUT1JJQ09fVklTSVRBX0NPTlNVTFRBUiJ9LHsiYXV0aG9yaXR5IjoiRklDSEFfVklTSVRBTlRFXy1fVU5JREFERVNfQUNFU1NPX0NPTlNVTFRBUiJ9LHsiYXV0aG9yaXR5IjoiRklDSEFfVklTSVRBTlRFX0FMVEVSQVIifSx7ImF1dGhvcml0eSI6IkZJQ0hBX1ZJU0lUQU5URV9DT05TVUxUQVIifSx7ImF1dGhvcml0eSI6IkZJQ0hBX1ZJU0lUQU5URV9JTVBSSU1JUiJ9LHsiYXV0aG9yaXR5IjoiR0VTVEFPX1VOSURBREVfTElWUk9fQ09OU1VMVEFSIn0seyJhdXRob3JpdHkiOiJMSVZST19QTEFOVEFPX0FMVEVSQVIifSx7ImF1dGhvcml0eSI6IkxJVlJPX1BMQU5UQU9fQ09OU1VMVEFSIn0seyJhdXRob3JpdHkiOiJMSVZST19QTEFOVEFPX0lNUFJJTUlSIn0seyJhdXRob3JpdHkiOiJMSVZST19QTEFOVEFPX0lOU0VSSVIifSx7ImF1dGhvcml0eSI6Ik1FRElDQU1FTlRPX0FVVE9SSVpBUiJ9LHsiYXV0aG9yaXR5IjoiTUVESUNBTUVOVE9fSU1QUklNSVIifSx7ImF1dGhvcml0eSI6Ik1FTlVfQ09OU1VMVEFSIn0seyJhdXRob3JpdHkiOiJNRU5VX0lNUFJJTUlSIn0seyJhdXRob3JpdHkiOiJNRU5VX1NDQVJfQ09OU1VMVEFSIn0seyJhdXRob3JpdHkiOiJNRU5VX1NDQVJfSU5TRVJJUiJ9LHsiYXV0aG9yaXR5IjoiTU9EVUxPX0NPTlNVTFRBUiJ9LHsiYXV0aG9yaXR5IjoiTU9EVUxPX0lNUFJJTUlSIn0seyJhdXRob3JpdHkiOiJNT1ZJTUVOVEFDQU9fLV9BTFRFUkFSX0RBVEFfTU9WX01BSVNfUVVFX0NJTkNPX0RJQVNfQUxURVJBUiJ9LHsiYXV0aG9yaXR5IjoiTU9WSU1FTlRBQ0FPX0FMVEVSQVIifSx7ImF1dGhvcml0eSI6Ik1PVklNRU5UQUNBT19BVVRPUklaQVIifSx7ImF1dGhvcml0eSI6Ik1PVklNRU5UQUNBT19DT05TVUxUQVIifSx7ImF1dGhvcml0eSI6Ik1PVklNRU5UQUNBT19FWENMVUlSIn0seyJhdXRob3JpdHkiOiJNT1ZJTUVOVEFDQU9fSU1QUklNSVIifSx7ImF1dGhvcml0eSI6Ik1PVklNRU5UQUNBT19JTlNFUklSIn0seyJhdXRob3JpdHkiOiJPQ09SUkVOQ0lBX1VOSURBREVfQUxURVJBUiJ9LHsiYXV0aG9yaXR5IjoiT0NPUlJFTkNJQV9VTklEQURFX0FVVE9SSVpBUiJ9LHsiYXV0aG9yaXR5IjoiT0NPUlJFTkNJQV9VTklEQURFX0NPTlNVTFRBUiJ9LHsiYXV0aG9yaXR5IjoiT0NPUlJFTkNJQV9VTklEQURFX0lNUFJJTUlSIn0seyJhdXRob3JpdHkiOiJQQURSQU9fVklTSVRBX0FMVEVSQVIifSx7ImF1dGhvcml0eSI6IlBBRFJBT19WSVNJVEFfQ09OU1VMVEFSIn0seyJhdXRob3JpdHkiOiJQQURSQU9fVklTSVRBX0VYQ0xVSVIifSx7ImF1dGhvcml0eSI6IlBBRFJBT19WSVNJVEFfSU1QUklNSVIifSx7ImF1dGhvcml0eSI6IlBBVE9MT0dJQV9DT05TVUxUQVIifSx7ImF1dGhvcml0eSI6IlBBVE9MT0dJQV9JTVBSSU1JUiJ9LHsiYXV0aG9yaXR5IjoiUEVSRklMX0NPTlNVTFRBUiJ9LHsiYXV0aG9yaXR5IjoiUEVSRklMX0lNUFJJTUlSIn0seyJhdXRob3JpdHkiOiJQRVNTT0FfQUxURVJBUiJ9LHsiYXV0aG9yaXR5IjoiUEVTU09BX0JJT01FVFJJQV9BTFRFUkFSIn0seyJhdXRob3JpdHkiOiJQRVNTT0FfQklPTUVUUklBX0FVVE9SSVpBUiJ9LHsiYXV0aG9yaXR5IjoiUEVTU09BX0JJT01FVFJJQV9DT05TVUxUQVIifSx7ImF1dGhvcml0eSI6IlBFU1NPQV9CSU9NRVRSSUFfSU5TRVJJUiJ9LHsiYXV0aG9yaXR5IjoiUEVTU09BX0NPTlNVTFRBUiJ9LHsiYXV0aG9yaXR5IjoiUEVTU09BX0lNUFJJTUlSIn0seyJhdXRob3JpdHkiOiJQRVNTT0FfSU5TRVJJUiJ9LHsiYXV0aG9yaXR5IjoiUE9TVE9TX0FMVEVSQVIifSx7ImF1dGhvcml0eSI6IlBPU1RPU19DT05TVUxUQVIifSx7ImF1dGhvcml0eSI6IlBPU1RPU19JTlNFUklSIn0seyJhdXRob3JpdHkiOiJQUkVTVEFET1JfU0VSVklDT19BTFRFUkFSIn0seyJhdXRob3JpdHkiOiJQUkVTVEFET1JfU0VSVklDT19DT05TVUxUQVIifSx7ImF1dGhvcml0eSI6IlBSRVNUQURPUl9TRVJWSUNPX0lNUFJJTUlSIn0seyJhdXRob3JpdHkiOiJQUkVTVEFET1JfU0VSVklDT19JTlNFUklSIn0seyJhdXRob3JpdHkiOiJSRUdSQV9WSVNJVEFfQUxURVJBUiJ9LHsiYXV0aG9yaXR5IjoiUkVHUkFfVklTSVRBX0NPTlNVTFRBUiJ9LHsiYXV0aG9yaXR5IjoiUkVHUkFfVklTSVRBX0VYQ0xVSVIifSx7ImF1dGhvcml0eSI6IlJFR1JBX1ZJU0lUQV9JTVBSSU1JUiJ9LHsiYXV0aG9yaXR5IjoiUkVHUkFfVklTSVRBX0lOU0VSSVIifSx7ImF1dGhvcml0eSI6IlJFTEFUT1JJT19FU1RBVElTVElDQV9BTEJFUkdBRE9fUFJFU0VOQ0FTX0NPTlNVTFRBUiJ9LHsiYXV0aG9yaXR5IjoiUkVMQVRPUklPX0VTVEFUSVNUSUNBX0FMQkVSR0FET19QUkVTRU5DQV9GSUxJQUNBT19DT05TVUxUQVIifSx7ImF1dGhvcml0eSI6IlJFTEFUT1JJT19FU1RBVElTVElDQV9BTkVYT19DT05TVUxUQVIifSx7ImF1dGhvcml0eSI6IlJFTEFUT1JJT19FU1RBVElTVElDQV9BVEVORElNRU5UT19NRURJQ09fQ09OU1VMVEFSIn0seyJhdXRob3JpdHkiOiJSRUxBVE9SSU9fRVNUQVRJU1RJQ0FfQVRFTkRJTUVOVE9fTUVESUNPX0lNUFJJTUlSIn0seyJhdXRob3JpdHkiOiJSRUxBVE9SSU9fRVNUQVRJU1RJQ0FfQVRJVklEQURFX0xBQk9SQUxfQ09OU1VMVEFSIn0seyJhdXRob3JpdHkiOiJSRUxBVE9SSU9fRVNUQVRJU1RJQ0FfQVRJVklEQURFX0xBQk9SQUxfSU1QUklNSVIifSx7ImF1dGhvcml0eSI6IlJFTEFUT1JJT19FU1RBVElTVElDQV9DT01QQU5IRUlST1NfTE9DQUxJWkFDQU9fQ09OU1VMVEFSIn0seyJhdXRob3JpdHkiOiJSRUxBVE9SSU9fRVNUQVRJU1RJQ0FfQ09NUEFOSEVJUk9TX0xPQ0FMSVpBQ0FPX0lNUFJJTUlSIn0seyJhdXRob3JpdHkiOiJSRUxBVE9SSU9fRVNUQVRJU1RJQ0FfQ1VTVE9ESUFET19TRU1fVklTSVRBX0NPTlNVTFRBUiJ9LHsiYXV0aG9yaXR5IjoiUkVMQVRPUklPX0VTVEFUSVNUSUNBX0pVUklESUNPX0FUSVZJREFERV9DT05TVUxUQVIifSx7ImF1dGhvcml0eSI6IlJFTEFUT1JJT19FU1RBVElTVElDQV9KVVJJRElDT19BVElWSURBREVfSU1QUklNSVIifSx7ImF1dGhvcml0eSI6IlJFTEFUT1JJT19FU1RBVElTVElDQV9KVVJJRElDT19JTkNJREVOQ0lBX0NPTlNVTFRBUiJ9LHsiYXV0aG9yaXR5IjoiUkVMQVRPUklPX0VTVEFUSVNUSUNBX0pVUklESUNPX1BST0NFU1NPX0NPTlNVTFRBUiJ9LHsiYXV0aG9yaXR5IjoiUkVMQVRPUklPX0VTVEFUSVNUSUNBX01FRElDQU1FTlRPX0NPTlNVTFRBUiJ9LHsiYXV0aG9yaXR5IjoiUkVMQVRPUklPX0VTVEFUSVNUSUNBX01FRElDQU1FTlRPX0lNUFJJTUlSIn0seyJhdXRob3JpdHkiOiJSRUxBVE9SSU9fRVNUQVRJU1RJQ0FfUEFUT0xPR0lBX0NPTlNVTFRBUiJ9LHsiYXV0aG9yaXR5IjoiUkVMQVRPUklPX0VTVEFUSVNUSUNBX1BBVE9MT0dJQV9JTVBSSU1JUiJ9LHsiYXV0aG9yaXR5IjoiUkVMQVRPUklPX0VTVEFUSVNUSUNBX1BPUFVMQUNBT19DQVJDRVJBUklBX0NPTlNVTFRBUiJ9LHsiYXV0aG9yaXR5IjoiUkVMQVRPUklPX0VTVEFUSVNUSUNBX1BPUFVMQUNBT19DQVJDRVJBUklBX0lNUFJJTUlSIn0seyJhdXRob3JpdHkiOiJSRUxBVE9SSU9fRVNUQVRJU1RJQ0FfUkVJTkNJREVOQ0lBX0NPTlNVTFRBUiJ9LHsiYXV0aG9yaXR5IjoiUkVMQVRPUklPX0VTVEFUSVNUSUNBX1JFSU5DSURFTkNJQV9JTVBSSU1JUiJ9LHsiYXV0aG9yaXR5IjoiUkVMQVRPUklPX0VTVEFUSVNUSUNBX1JFU1VNT19QT1BVTEFDQU9fQ0FSQ0VSQVJJQV9DT05TVUxUQVIifSx7ImF1dGhvcml0eSI6IlJFTEFUT1JJT19FU1RBVElTVElDQV9SRVNVTU9fUE9QVUxBQ0FPX0NBUkNFUkFSSUFfSU1QUklNSVIifSx7ImF1dGhvcml0eSI6IlJFTEFUT1JJT19FU1RBVElTVElDQV9TSU5URVRJQ09fRU5UUkFEQV9ERUxFR0FDSUFfQ09OU1VMVEFSIn0seyJhdXRob3JpdHkiOiJSRUxBVE9SSU9fRVNUQVRJU1RJQ0FfU0lOVEVUSUNPX0VOVFJBREFfREVMRUdBQ0lBX0lNUFJJTUlSIn0seyJhdXRob3JpdHkiOiJSRUxBVE9SSU9fRVNUQVRJU1RJQ0FfU0lOVEVUSUNPX1NBSURBX1RFTVBPUkFSSUFfQ09OU1VMVEFSIn0seyJhdXRob3JpdHkiOiJSRUxBVE9SSU9fRVNUQVRJU1RJQ0FfU0lOVEVUSUNPX1NBSURBX1RFTVBPUkFSSUFfSU1QUklNSVIifSx7ImF1dGhvcml0eSI6IlJFTEFUT1JJT19FU1RBVElTVElDQV9URVJNSU5BTF9FTlRSQURBX1NBSURBX0NPTlNVTFRBUiJ9LHsiYXV0aG9yaXR5IjoiUkVMQVRPUklPX0VTVEFUSVNUSUNBX1RFUk1JTkFMX0VOVFJBREFfU0FJREFfSU1QUklNSVIifSx7ImF1dGhvcml0eSI6IlJFTEFUT1JJT19FU1RBVElTVElDQV9URVJNSU5BTF9MT0dfQ09OU1VMVEFSIn0seyJhdXRob3JpdHkiOiJSRUxBVE9SSU9fRVNUQVRJU1RJQ0FfVEVSTUlOQUxfTE9HX0lNUFJJTUlSIn0seyJhdXRob3JpdHkiOiJSRUxBVE9SSU9fRVNUQVRJU1RJQ0FfVFJBTlNGRVJFTkNJQV9ERV9VTklEQURFX0NPTlNVTFRBUiJ9LHsiYXV0aG9yaXR5IjoiUkVMQVRPUklPX0VTVEFUSVNUSUNBX1RSQU5TRkVSRU5DSUFfREVfVU5JREFERV9JTVBSSU1JUiJ9LHsiYXV0aG9yaXR5IjoiUkVMQVRPUklPX0VTVEFUSVNUSUNBX1ZBQ0lOQV9DT05TVUxUQVIifSx7ImF1dGhvcml0eSI6IlJFTEFUT1JJT19FU1RBVElTVElDQV9WQUNJTkFfSU1QUklNSVIifSx7ImF1dGhvcml0eSI6IlJFTEFUT19GQVRPX0FMVEVSQVIifSx7ImF1dGhvcml0eSI6IlJFTEFUT19GQVRPX0NPTlNVTFRBUiJ9LHsiYXV0aG9yaXR5IjoiUkVMQVRPX0ZBVE9fSU5TRVJJUiJ9LHsiYXV0aG9yaXR5IjoiUkVTUE9OU0FWRUxKVVJJRElDT19DT05TVUxUQVIifSx7ImF1dGhvcml0eSI6IlNFVE9SX0FMVEVSQVIifSx7ImF1dGhvcml0eSI6IlNFVE9SX0FVVE9SSVpBUiJ9LHsiYXV0aG9yaXR5IjoiU0VUT1JfQ09OU1VMVEFSIn0seyJhdXRob3JpdHkiOiJTRVRPUl9FWENMVUlSIn0seyJhdXRob3JpdHkiOiJTRVRPUl9JTVBSSU1JUiJ9LHsiYXV0aG9yaXR5IjoiU0VUT1JfSU5TRVJJUiJ9LHsiYXV0aG9yaXR5IjoiU0lTVEVNQV9DT05TVUxUQVIifSx7ImF1dGhvcml0eSI6IlNJU1RFTUFfSU1QUklNSVIifSx7ImF1dGhvcml0eSI6IlRJUE9fTElWUk9fQ09OU1VMVEFSIn0seyJhdXRob3JpdHkiOiJVTklEQURFX0NPTlNVTFRBUiJ9LHsiYXV0aG9yaXR5IjoiVU5JREFERV9JTVBSSU1JUiJ9LHsiYXV0aG9yaXR5IjoiVU5JREFERV9JTlNFUklSIn0seyJhdXRob3JpdHkiOiJVU1VBUklPIn0seyJhdXRob3JpdHkiOiJVU1VBUklPU19NSU5IQVNfQUNPRVNfQ09OU1VMVEFSIn0seyJhdXRob3JpdHkiOiJVU1VBUklPU19NSU5IQVNfUk9USU5BU19BTFRFUkFSIn0seyJhdXRob3JpdHkiOiJVU1VBUklPU19NSU5IQVNfUk9USU5BU19DT05TVUxUQVIifSx7ImF1dGhvcml0eSI6IlVTVUFSSU9fQ09OU1VMVEFSIn0seyJhdXRob3JpdHkiOiJVU1VBUklPX0lNUFJJTUlSIn0seyJhdXRob3JpdHkiOiJWQUNJTkFfQVVUT1JJWkFSIn0seyJhdXRob3JpdHkiOiJWQUNJTkFfSU1QUklNSVIifSx7ImF1dGhvcml0eSI6IlZJU0lUQU5URV8tX0FCQV9SRUxBQ09FU19BTFRFUkFSIn0seyJhdXRob3JpdHkiOiJWSVNJVEFOVEVfQUxURVJBUiJ9LHsiYXV0aG9yaXR5IjoiVklTSVRBTlRFX0NPTlNVTFRBUiJ9LHsiYXV0aG9yaXR5IjoiVklTSVRBTlRFX0lNUFJJTUlSIn0seyJhdXRob3JpdHkiOiJWSVNJVEFOVEVfSU5TRVJJUiJ9LHsiYXV0aG9yaXR5IjoiVklTSVRBTlRFX1VOSURBREVfQUxURVJBUiJ9LHsiYXV0aG9yaXR5IjoiVklTSVRBTlRFX1VOSURBREVfQ09OU1VMVEFSIn0seyJhdXRob3JpdHkiOiJWSVNJVEFOVEVfVU5JREFERV9JTVBSSU1JUiJ9LHsiYXV0aG9yaXR5IjoiVklTSVRBTlRFX1VOSURBREVfSU5TRVJJUiJ9LHsiYXV0aG9yaXR5IjoiVklTSVRBU19DT05TVUxUQVIifSx7ImF1dGhvcml0eSI6IlZJU0lUQVNfSU1QUklNSVIifSx7ImF1dGhvcml0eSI6IlZJU0lUQV9BVlVMU0FfTlVWSUdfQUxURVJBUiJ9LHsiYXV0aG9yaXR5IjoiVklTSVRBX0FWVUxTQV9OVVZJR19BVVRPUklaQVIifSx7ImF1dGhvcml0eSI6IlZJU0lUQV9BVlVMU0FfTlVWSUdfQ09OU1VMVEFSIn0seyJhdXRob3JpdHkiOiJWSVNJVEFfQVZVTFNBX05VVklHX0VYQ0xVSVIifSx7ImF1dGhvcml0eSI6IlZJU0lUQV9BVlVMU0FfTlVWSUdfSU1QUklNSVIifSx7ImF1dGhvcml0eSI6IlZJU0lUQV9BVlVMU0FfTlVWSUdfSU5TRVJJUiJ9LHsiYXV0aG9yaXR5IjoiVklTSVRBX0dFUkFMX0NPTlNVTFRBUiJ9LHsiYXV0aG9yaXR5IjoiVklTSVRBX0dFUkFMX0lNUFJJTUlSIn0seyJhdXRob3JpdHkiOiJWSVNJVEFfU0FJREFfTE9URV9BVVRPUklaQVIifSx7ImF1dGhvcml0eSI6IlZJU0lUQV9TRU5IQV9DT05TVUxUQVIifSx7ImF1dGhvcml0eSI6IlZJU0lUQV9TRU5IQV9JTVBSSU1JUiJ9XSwidXN1YXJpb19pZCI6MzEwNCwidXN1YXJpb19ub21lIjoiRUxJU0FOR0VMQSBNQVJJQSBEQSBTSUxWQSBIRUxDSUFTIiwibW9kdWxvX2lkIjoxLCJpYXQiOjE3NzA4OTg0MjAsImV4cCI6MTc3MDkwOTIyMH0.bJd3GIiMe7hCHzpL05uKzlP4N7dMaGCAP7BGy5ZaL6c"

# Cabeçalhos da requisição
headers = {
    "Authorization": TOKEN,
    "Accept": "application/json"
}

# Loop pelos IDs
for custodiado_id in ids:
    
    
    
    url = f"http://172.18.4.46:8080/sispen_api/fichas-custodiados/{custodiado_id}/visitantes?page=0&size=32&sort=pessoa.nome%2Casc"
    
    try:
        # Fallback: buscaRapida
        payload = {
                    "nome": str(custodiado_id),
                    "tiposPessoa": [1,2,3,4,7,8],
                    "unidadesAcesso": [170,157]
                 }
        
        resp_fallback = requests.post("http://172.18.4.46:8080/sispen_api/pessoa/buscaRapida",headers=headers, json=payload)
        
        if resp_fallback.status_code == 200:
                dados = resp_fallback.json()

        # Se vier lista, pega o primeiro item
        if isinstance(dados, list) and dados:
                dados = dados[0]

        foto = dados.get("foto", "")

        if "/PRESO/" in foto:
                novo_id = foto.split("/PRESO/")[1].split("/")[0]
                url = f"http://172.18.4.46:8080/sispen_api/fichas-custodiados/{novo_id}/visitantes?page=0&size=32&sort=pessoa.nome%2Casc"
                response = requests.get(url, headers=headers)
                
                if response.status_code == 200:
                    data = response.json()
                    entities = data.get("entities", [])
                    if not entities:
                        print(f"{custodiado_id} - Sem cadastro")
                    else:
                        visitantes_validos = [v for v in entities if v.get("dataUltimaVisita")]
                        if not visitantes_validos:
                            print(f"{custodiado_id} - Sem cadastro")
                        else:
                            ultimo = max(visitantes_validos, key=lambda x: x["dataUltimaVisita"])
                            nome = ultimo["pessoa"]["nome"].strip()
                            data_epoch = ultimo["dataUltimaVisita"]
                            data_legivel = datetime.fromtimestamp(data_epoch / 1000).strftime("%d/%m/%Y")
                            print(f"{custodiado_id} (real: {novo_id}) - {nome} - Última visita: {data_legivel}")

                else:
                    print(f"{custodiado_id} - Erro {response.status_code}")

        #############
    except Exception as e:
        print(f"{custodiado_id} - Erro na requisição: {e}")

    # Delay de 1 segundo entre requisições 
    time.sleep(1)


Total de IDs capturados: 190
[585934, 568860, 193264, 582838, 569056, 578117, 418912, 371867, 327575, 556302]
585934 - Erro na requisição: name 'dados' is not defined
568860 - Erro na requisição: name 'dados' is not defined
193264 - Erro na requisição: name 'dados' is not defined
582838 - Erro na requisição: name 'dados' is not defined
569056 - Erro na requisição: name 'dados' is not defined
578117 - Erro na requisição: name 'dados' is not defined
418912 - Erro na requisição: name 'dados' is not defined


KeyboardInterrupt: 

: 